# Nemotron-3.5 Lightning Text2SQL LoRA Fine-Tuning using Megatron Bridge

Data prep, checkpoint conversion, LoRA fine-tuning, then merge and export, using [NeMo Megatron-Bridge](https://github.com/NVIDIA-NeMo/Megatron-Bridge).

This model fits on a single node, and only the training step needs a GPU.

## Hardware

Set `n_devices` to the number of GPUs you have; expert parallelism follows it. Measured on 80GB H100s, training one epoch at the defaults below:

| GPUs | Peak memory/GPU | One epoch |
| --- | --- | --- |
| 1 | ~79 GB | ~60 min |
| 2 | ~51 GB | ~34 min |
| 4 | ~35 GB | ~18 min |
| 8 | ~27 GB | ~8 min |

On one GPU the training cell sets `REDUCE_MTP_HEADS=1` for you, which is what makes the model fit. Use two GPUs if you have them.

You also need about 130 GB of disk, plus room for the merged export.

## Prerequisites

- At least one H100 80GB (or similar) GPU.
- The Nemotron-3.5 Lightning checkpoint downloaded to a local directory.
- A Hugging Face token in `$HF_TOKEN`, used to download BIRD.

## Launching the container

Launch the NeMo container with this notebook directory mounted. Adjust `--gpus` to match your machine.

```bash
CONTAINER="nvcr.io/nvidian/nemo:26.08"

docker container run --gpus all -it --rm \
  -e HF_TOKEN -e HF_HOME=/root/.cache/huggingface \
  -v $HOME/.cache:/root/.cache \
  -v "$(pwd)":/workspace/notebook \
  --shm-size=16g --net=host --ipc=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  "${CONTAINER}" bash
```

Run the rest of the cells from inside the container.

---
## Configuration

Edit the values below to match your environment. This is the only cell you should need to change.

In [ ]:
import os

n_devices = 1  # How many GPUs to train on? See the hardware table above.
max_seq_len = 2048  # Max sequence length for training (in tokens).
hf_model = "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16"  # Hub ID, or a local path.
base_model_path = "/workspace/storage/nemotron-3.5-lightning/base"  # Converted checkpoint goes here.
experiment_name = "lightning35-text2sql-lora"
dataprep_output_dir = os.path.abspath("results/dataprep")
train_output_dir = os.path.abspath("results/trained")

expert_parallel = n_devices  # Split the experts across your GPUs.

%env MODEL_ID=$hf_model
%env MAX_SEQ_LEN=$max_seq_len
%env DATAPREP_OUTPUT_DIR=$dataprep_output_dir

---
## Step 1: Data Preparation

Build `results/dataprep/training.jsonl` from the [BIRD](https://huggingface.co/datasets/xu3kev/BIRD-SQL-data-train) dataset, using both the no-reasoning and reasoning splits. Records are formatted with the model's chat template, so training matches what the model sees at inference.

Set `MAX_TRAIN_SAMPLES` to a small number for a quick smoke run.

In [ ]:
! MAX_TRAIN_SAMPLES=0 python dataprep.py

### Sanity Check

Verify the training data was created, and set the environment variables the remaining steps need.

In [ ]:
training_jsonl = os.path.join(dataprep_output_dir, "training.jsonl")
assert os.path.exists(training_jsonl), f"Expected training data at '{training_jsonl}'. Run the data prep cell first."

with open(training_jsonl) as f:
    print("Training examples:", sum(1 for _ in f))

# Set environment variables for the subsequent %%bash cells.
%env N_DEVICES=$n_devices
%env EP=$expert_parallel
%env MAX_SEQ_LEN=$max_seq_len
%env HF_MODEL=$hf_model
%env DATASET_DIR=$dataprep_output_dir
%env MEGATRON_MODEL_PATH=$base_model_path
%env EXPERIMENT_NAME=$experiment_name
%env TRAINING_OUTPUT_DIR=$train_output_dir

---
## Step 2: Model Conversion

Megatron-Bridge trains from its own checkpoint format, so convert the Hugging Face weights once. Runs on CPU.

The result is `torch_dist` format, which reshards on load, so you can convert once and train on any number of GPUs. Re-running is a no-op once the checkpoint exists.

In [ ]:
%%bash
set -euo pipefail
python convert.py

### Sanity Check

In [ ]:
%%bash
test -e "$MEGATRON_MODEL_PATH/latest_checkpointed_iteration.txt" || { echo "No converted checkpoint at $MEGATRON_MODEL_PATH"; exit 1; }
ls "$MEGATRON_MODEL_PATH"

---
## Step 3: LoRA Fine-Tuning

The only step that needs GPUs. The shipped Megatron-Bridge recipe supplies the LoRA target modules and the rest of the model-specific config; `train.py` points it at your paths and GPU count.

Training uses packed sequences: examples are concatenated into dense `seq_length`-token blocks rather than padded individually.

> The first iteration takes a minute or two while CUDA graphs are captured and the MoE warms up. It is not a hang.

In [ ]:
%%bash
set -euo pipefail

# Reduces allocator fragmentation. Needed on 1 GPU, harmless otherwise.
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# On a single GPU the model only fits with one MTP head instead of the recipe's two.
if [ "$N_DEVICES" -eq 1 ]; then
  export REDUCE_MTP_HEADS=1
fi

torchrun --nproc-per-node="$N_DEVICES" train.py

### Sanity Check

Confirm a LoRA adapter checkpoint was saved.

In [ ]:
%%bash
RUN_DIR="$TRAINING_OUTPUT_DIR/$EXPERIMENT_NAME"
test -e "$RUN_DIR/latest_checkpointed_iteration.txt" || { echo "No saved adapter under $RUN_DIR"; exit 1; }
ls "$RUN_DIR"

---
## Step 4: Merge the LoRA Adapter and Export to Hugging Face

Merge the adapter back into the base weights and write a standard Hugging Face checkpoint, ready for inference or upload. Runs on CPU. The cell below finds the latest training checkpoint automatically.

In [ ]:
run_dir = os.path.join(train_output_dir, experiment_name)
iter_file = os.path.join(run_dir, "latest_checkpointed_iteration.txt")
assert os.path.exists(iter_file), f"'{iter_file}' not found — run the training step first."

with open(iter_file) as f:
    latest_iter = int(f.read().strip())

lora_checkpoint = os.path.join(run_dir, f"iter_{latest_iter:07d}")
merge_output_dir = lora_checkpoint + "_merged_hf"
print("Merge output:", merge_output_dir)

%env LORA_CHECKPOINT=$lora_checkpoint
%env MERGE_OUTPUT_DIR=$merge_output_dir

In [ ]:
%%bash
set -euo pipefail

torchrun --nproc-per-node=1 \
  /opt/Megatron-Bridge/examples/peft/merge_lora.py \
  --lora-checkpoint "$LORA_CHECKPOINT" \
  --hf-model-path "$HF_MODEL" \
  --output "$MERGE_OUTPUT_DIR" \
  --cpu

---
## Step 5: Try the Fine-Tuned Model with vLLM

vLLM supports this architecture natively, so you can serve the merged checkpoint directly. Prompt it the way data prep formatted the training examples: schema, blank line, question, then optional evidence. Use `enable_thinking=False` for direct SQL.

For serving, run `vllm serve $MERGE_OUTPUT_DIR` and use the OpenAI-compatible API.

In [ ]:
%%writefile try_model.py
import os

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

SCHEMA = """CREATE TABLE employees (
  id INT, name TEXT, dept_id INT, salary INT, hire_date DATE
);
CREATE TABLE departments (
  dept_id INT, dept_name TEXT, location TEXT
);"""
QUESTION = "Which department has the highest average salary?"


def main():
    model_path = os.environ["MERGE_OUTPUT_DIR"]
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

    prompt = tok.apply_chat_template(
        [{"role": "system", "content": ""},
         {"role": "user", "content": f"{SCHEMA}\n\n{QUESTION}"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

    llm = LLM(model=model_path, tokenizer=model_path, trust_remote_code=True,
              dtype="bfloat16", max_model_len=8192, gpu_memory_utilization=0.92,
              enforce_eager=True)
    out = llm.generate([prompt], SamplingParams(temperature=0.0, max_tokens=256))
    print(out[0].outputs[0].text.strip())


if __name__ == "__main__":  # required: vLLM spawns worker processes
    main()


In [ ]:
! python try_model.py
